# Parlance SLM — QLoRA Fine-Tuning

Fine-tunes **Qwen 2.5 0.5B Instruct** into two merged grammar-coaching models (Spanish & French) via QLoRA.

### GPU: Google Colab in the browser (not your Mac)
All training runs on **Google’s cloud T4 GPU** in Colab. Your Mac is only used to upload data and download the finished models — it does not train locally.

### Setup
1. **Runtime → Change runtime type → T4 GPU** (required).
2. Run **Step 1** (install), then **Step 2** (clone repo + upload the four JSONL files from your Mac).

### Train (~15–30 min per language on Colab’s T4)
Steps 4–5 call `finetune_slm.py`. Output: `models/parlance-es/` and `models/parlance-fr/`. Zip and download in Step 7.

**Open in browser:** [colab.research.google.com — this notebook](https://colab.research.google.com/github/Montrez/ParlanceApp/blob/main/training/Parlance_FineTune.ipynb)

In [ ]:
# Step 1: Install dependencies
!pip install -q torch transformers peft datasets accelerate bitsandbytes trl

In [ ]:
# Step 2: Clone repo (browser Colab) and verify data paths
import os
from pathlib import Path

if not Path("finetune_slm.py").exists():
    if not Path("ParlanceApp/training/finetune_slm.py").exists():
        print("Cloning ParlanceApp from GitHub...")
        !git clone --depth 1 https://github.com/Montrez/ParlanceApp.git
    os.chdir("ParlanceApp/training")

assert Path("finetune_slm.py").exists(), "finetune_slm.py missing after setup"

for sub in ("data/spanish", "data/french", "models", "checkpoints"):
    Path(sub).mkdir(parents=True, exist_ok=True)

required = {
    "data/spanish/train.jsonl": 1275,
    "data/spanish/valid.jsonl": 142,
    "data/french/train.jsonl": 1196,
    "data/french/valid.jsonl": 133,
}

missing = [p for p in required if not Path(p).exists()]
if missing:
    from google.colab import files
    import zipfile

    print("Upload parlance_training_data.zip (from training/ on your Mac) — or each JSONL separately.")
    up = files.upload()
    for name, data in up.items():
        if name.endswith(".zip"):
            zpath = Path(name)
            zpath.write_bytes(data)
            with zipfile.ZipFile(zpath) as zf:
                zf.extractall(".")
            print(f"  extracted {name}")
        else:
            dest = Path("data/spanish" if "spanish" in name else "data/french") / name
            if name in ("train.jsonl", "valid.jsonl"):
                for lang in ("spanish", "french"):
                    if (Path(f"data/{lang}") / name).as_posix() in missing:
                        dest = Path(f"data/{lang}/{name}")
                        break
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(data)
            print(f"  saved {dest}")
    missing = [p for p in required if not Path(p).exists()]
    while missing:
        print(f"\nStill missing: {missing[0]} — upload that file:")
        up = files.upload()
        for _, data in up.items():
            Path(missing[0]).parent.mkdir(parents=True, exist_ok=True)
            Path(missing[0]).write_bytes(data)
        missing = [p for p in required if not Path(p).exists()]

for rel, expected in required.items():
    n = sum(1 for line in open(rel) if line.strip())
    ok = "ok" if n == expected else f"expected {expected}"
    print(f"  {rel}: {n} ({ok})")

print(f"\nWorking directory: {Path.cwd().resolve()}")
print("Ready to train.")

In [ ]:
# Step 3: Check GPU
import torch

assert torch.cuda.is_available(), "Enable Runtime → T4 GPU before training."

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Base model: Qwen/Qwen2.5-0.5B-Instruct")

In [ ]:
# Step 4: Train Spanish SLM (~15–30 min on T4)
!python finetune_slm.py --lang es --epochs 3

In [ ]:
# Step 5: Train French SLM (~15–30 min on T4)
!python finetune_slm.py --lang fr --epochs 3

In [ ]:
# Step 6: Quick smoke test (Spanish model)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "models/parlance-es"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)

messages = [
    {"role": "system", "content": "You are a Spanish grammar coach for interpreter training. Analyze the learner's sentence at CEFR level B1. Respond with a JSON object containing: status, grammar_rule, explanation, correction, next_level_alt, target_level_alt, and tip. All example sentences must be in Spanish."},
    {"role": "user", "content": 'Analyze this Spanish sentence: "Ayer yo iba al supermercado y compré muchas cosas."'},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
# Step 7: Zip and download merged models
!zip -rq parlance-es.zip models/parlance-es/
!zip -rq parlance-fr.zip models/parlance-fr/

from google.colab import files

print("Downloading Spanish model...")
files.download("parlance-es.zip")
print("Downloading French model...")
files.download("parlance-fr.zip")

In [ ]:
# On your Mac, unzip into ParlanceApp/training/models/
#   parlance-es/  and  parlance-fr/